In [1]:
import sys
import os
import numpy as np
import torch
from torch import nn
from tqdm import tqdm

from einops.layers.torch import Rearrange
from einops import rearrange, repeat
from fvcore.nn import FlopCountAnalysis, flop_count_table


In [2]:
sys.path.append(os.path.abspath(os.path.join('..', 'modules')))
from Trainer import Trainer

In [3]:
config = {
    # image h w c
    'height': 72,
    'width': 144,
    'channels': 2,
    # video length in and out
    'T1': 1,    # 5 later.
    'MaxT2': 5,
    'FG': 24,   # frames in a frame group
    # patch size
    'patch_size': 8,
    # transformer
    'heads': 8,
    'dim_head': 32,
    'dropout': 0.0,
    'attn_dropout': 0.0,
    'drop_path': 0.0,
    'scale_dim': 4,
    # layers
    'attLayers' : 1,
    'zeusLayers': 6,
}

In [4]:
trainer = Trainer(config)

In [5]:
x = torch.rand(1, 24, 2, 72, 144)
output = trainer.model(x)
print(output.shape)  # [B, T, C, H, W]
# # Calculate FLOPs
flops = FlopCountAnalysis(trainer.model, x)
print(f'Number of flops: {flop_count_table(flops)}')


torch.Size([1, 24, 2, 72, 144])
Number of flops: | module                              | #parameters or shape   | #flops    |
|:------------------------------------|:-----------------------|:----------|
| model                               | 7.947M                 | 35.341G   |
|  to_patch_embedding.1               |  16.512K               |  63.701M  |
|   to_patch_embedding.1.weight       |   (128, 128)           |           |
|   to_patch_embedding.1.bias         |   (128,)               |           |
|  blocks                             |  7.913M                |  35.199G  |
|   blocks.0                          |   1.319M               |   5.866G  |
|    blocks.0.ts_temporal_transformer |    0.33M               |    1.329G |
|    blocks.0.ts_space_transformer    |    0.33M               |    1.604G |
|    blocks.0.st_space_transformer    |    0.33M               |    1.604G |
|    blocks.0.st_temporal_transformer |    0.33M               |    1.329G |
|   blocks.1               